 # Prefrence based Alignment Training
 Align or train a modal with human expectation or with human prefer data & to get a model that is safe, helpful and honest. After the prefrence based training model doesn't just produce accurat outputs, it behaves according to the user's prefrences, values and intentions.


dataset : argilla/ultrafeedback-binarized-prefrences-cleaned
dataset : xinlai/Math-Step-DPO-10K

####### dataset prompt initial_reason_steps chosen rejected full_chosen

Base Model -> Non Isntruction Fine Tuned Model -> Instruction FineTuned Model-> Alignment Finetuned Model

In [ ]:
base_model_path= "/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
instruction_model_path = "/content/drive/MyDrive/LLM/LLM_IFT_QLORA/checkpoint-12"

In [89]:
from  transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(instruction_model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [90]:
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

In [ ]:
prompt="Tell me the best 5 AI websites for web design"

In [ ]:
input= tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **input,
    max_new_tokens=350,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

Tell me the best 5 AI websites for web designers.
The best 5 AI websites for web designers.
AI is a very powerful tool that can help you in your work. It can do everything from creating beautiful designs to optimizing your website. There are many different types of AI, but the most common ones are machine learning and natural language processing. Machine learning is used to train computers to learn new things without being explicitly programmed. This means that they can make decisions based on their own experience. Natural language processing is used to understand what people are saying and then use that information to make decisions.
There are many different ways to use AI in your work. You can use it to create beautiful designs or optimize your website. You can also use it to analyze data and make predictions about future events. The best way to use AI is to find a company that specializes in this type of technology. They will be able to help you with all of your ne

In [ ]:
from trl import DPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
# Load dataset

dataset = load_dataset("csv",data_files="/content/drive/MyDrive/LLM/Datasets/pharmaceutical_preference_dataset.csv")["train"]


In [ ]:
dataset[0]

{'prompt': 'What is the mechanism of action of Metformin in treating type 2 diabetes?',
 'chosen': 'Metformin works primarily by inhibiting hepatic gluconeogenesis through the activation of AMP-activated protein kinase (AMPK). It reduces glucose production in the liver, improves insulin sensitivity in peripheral tissues, and decreases intestinal glucose absorption. It does not stimulate insulin secretion, so it carries a low risk of hypoglycemia.',
 'rejected': 'Metformin is a pill that lowers blood sugar. It works in the stomach and helps diabetic patients feel better. Doctors prescribe it a lot because it is cheap and available everywhere. Patients should take it with food to avoid side effects.'}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

get_peft_model() -> Create a New LoRA during traiing
PeftModel.from_pretrained() --> Load a already trained LoRA for infrence or further training

In [ ]:
# Loara config
lora_config=LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(base_model_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Load Instruction Lora and merge
model= PeftModel.from_pretrained(base_model, instruction_model_path)

In [ ]:
model =model.merge_and_unload()

In [ ]:
peft_model_lora = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from trl import DPOTrainer, DPOConfig

In [ ]:
dpo_args= DPOConfig(
    output_dir="/content/drive/MyDrive/LLM/LLM_AFT_QLORA",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    beta=0.1,
    loss_type="sigmoid",
    remove_unused_columns=False,
)

In [ ]:
trainer =DPOTrainer(
    model=peft_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,0.677384
20,0.669533


TrainOutput(global_step=20, training_loss=0.6734583854675293, metrics={'train_runtime': 26.0958, 'train_samples_per_second': 0.766, 'train_steps_per_second': 0.766, 'total_flos': 39115076861952.0, 'train_loss': 0.6734583854675293})

In [91]:
prompt="What is the mechanism of action of Metformin in treating type 2 diabetes?"

In [ ]:
NoN_Istruction_Finetuning_Model_path= "/content/drive/MyDrive/LLM/LLM_NFT_QLORA/checkpoint-8"

In [ ]:
NoN_Istruction_Finetuning_Model= AutoModelForCausalLM.from_pretrained(NoN_Istruction_Finetuning_Model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [92]:
input= tokenizer(prompt, return_tensors="pt").to("cuda")

# Output from Base Model

In [ ]:
outputs = base_model.generate(
    **input,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

What is the mechanism of action of Metformin in treating type 2 diabetes?
    }
        $1980000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


# Output from NIFT Model

In [ ]:
outputs = NoN_Istruction_Finetuning_Model.generate(
    **input,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

What is the mechanism of action of Metformin in treating type 2 diabetes?
What are the side effects of Metformin?
What are the contraindications for Metformin?
What are the precautions for Metformin?
What are the interactions of Metformin?
Metformin is a drug that is used to treat type 2 diabetes. It works by reducing the amount of sugar (glucose) produced by the liver and by increasing the amount of insulin produced by the pancreas.



# Output from Instruction FT Model

In [ ]:
outputs = instruction_model.generate(
    **input,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

What is the mechanism of action of Metformin in treating type 2 diabetes?
What are the side effects of Metformin?
What are the contraindications for Metformin?
How to take Metformin?
Metformin is a drug used to treat type 2 diabetes. It works by lowering blood sugar levels. The most common side effect of Metformin is weight gain. Other side effects include nausea, vomiting, and constipation.
Metformin is an oral medication that is used


# Output from Alingment Fine Tuned Model

In [93]:
Alignment_Finetuning_Model_path= "/content/drive/MyDrive/LLM/LLM_AFT_QLORA/checkpoint-20"

In [94]:
Alignment_Finetuning_Model= AutoModelForCausalLM.from_pretrained(Alignment_Finetuning_Model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [97]:
input= tokenizer(prompt, return_tensors="pt")

In [98]:
outputs = Alignment_Finetuning_Model.generate(
    **input,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [99]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

What is the mechanism of action of Metformin in treating type 2 diabetes?
What are the side effects of Metformin?
How does Metformin work?
Is Metformin safe to use?
Does Metformin cause weight loss?
Metformin is a drug that helps treat type 2 diabetes. It works by lowering blood sugar levels and increasing insulin production.
Metformin is an oral medication that is used to treat type 2 diabetes. It works by decreasing the amount of gl
